In [1]:
:set -XRankNTypes
:set -XExistentialQuantification
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XScopedTypeVariables
:set -XDeriveFunctor
import Data.List (foldl')
print "Setup done."
putStrLn "Setup done."

"Setup done."

Setup done.

# ∀ Лемма Енеды в Haskell

> *«Лемма Енеды — это не просто результат теории категорий. Это способ думать о функторах.» — Том Лайнстер*

Мы уже изучили функторы, естественные преобразования, монады и даже F-алгебры. Теперь пора подняться на уровень выше: **лемма Енеды** — один из глубочайших результатов теории категорий.

**Что говорит лемма:** множество всех естественных преобразований из Hom-функтора `Hom(a, -)` в функтор `F` находится в точном соответствии с `F(a)`:

```
Nat(Hom(a,-), F)  ≅  F(a)
```

В Haskell это выражается как изоморфизм типов:

```haskell
(forall b. (a -> b) -> f b)  ~  f a
```

**Почему это важно на практике:**
- Оптимизация `fmap` — отложить вычисления до последнего момента
- Свободный функтор Coyoneda — дать функтор любому типу
- Представимые функторы — описать функторы через индекс

![Изоморфизм Енеды](../diagrams/yoneda/yoneda_iso.svg)

❖Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | Yoneda: newtype и изоморфизм |  |
| 2 | Functor для Yoneda |  |
| 3 | Coyoneda: свободный функтор |  |
| 4 | Практика: ускорение fmap через Yoneda |  |
| 5 | Представимые функторы |  |
| 6 | Связь с расширениями Кана |  |

![Схема Coyoneda](../diagrams/yoneda/yoneda_coyoneda.svg)

## 🔗 Hom-функтор: основа леммы

**Hom(a, -)** — это функтор, который преобразует каждый объект `b` в множество стрелок из `a` в `b`. В Haskell:

```haskell
Hom(a, b)  =  a -> b   -- функции из a в b
Hom(a, -)  =  (->) a   -- функциональный тип, Functor
```

Стрелка `f :: b -> c` **действует** на Hom-множествах через последовательную композицию:

```haskell
-- это просто instance Functor ((->) a)
fmap f g = f . g
```

![Hom-функтор](../diagrams/yoneda/yoneda_hom.svg)

Таким образом, `(->) a` в Haskell — это и есть Hom-функтор `Hom(a,-)` в категории **Hask**.
Лемма Енеды говорит: естественные преобразования из `Hom(a,-)` в `F` ― это ровно элементы `F a`.

## Yoneda: newtype и изоморфизм

`Yoneda F a = forall b. (a -> b) -> F b` изоморфно `F a`.

Это значит: если у нас есть значение `forall b. (a -> b) -> F b`, то мы можем получить `F a` и наоборот.

In [2]:
-- Yoneda newtype: two type params, forall in field (safe)
newtype Yoneda f a = Yoneda { runYoneda :: forall b. (a -> b) -> f b }

-- toYoneda: F a -> Yoneda F a
-- If we have fa :: f a, then runYoneda (toYoneda fa) f = fmap f fa
toYoneda :: Functor f => f a -> Yoneda f a
toYoneda fa = Yoneda (\f -> fmap f fa)

-- fromYoneda: Yoneda F a -> F a  
-- Apply identity function
fromYoneda :: Yoneda f a -> f a
fromYoneda y = runYoneda y id

-- Test with Maybe
yMaybe :: Yoneda Maybe Int
yMaybe = toYoneda (Just 42)

-- Run with (+1): should give Just 43
result1 :: Maybe Int
result1 = runYoneda yMaybe (+1)

-- Round-trip: toYoneda then fromYoneda should give original
result2 :: Maybe Int
result2 = fromYoneda (toYoneda (Just 99))

print result1
print result2


Line 7: Avoid lambda using `infix`
Found:
(\ f -> fmap f fa)
Why not:
(`fmap` fa)

Just 43

Just 99

## Functor для Yoneda

Ключевое: `fmap` для Yoneda не требует `Functor f`! Он просто композирует функции.

In [3]:
-- Functor instance for Yoneda (does NOT require Functor f!)
-- This is the power: fmap just composes functions
instance Functor (Yoneda f) where
  fmap g (Yoneda y) = Yoneda (\h -> y (h . g))

-- Test: fmap over Yoneda Maybe Int
yMaybe2 :: Yoneda Maybe Int
yMaybe2 = toYoneda (Just 10)

-- fmap (+5) defers the application
mapped :: Yoneda Maybe Int
mapped = fmap (+5) yMaybe2

-- Extract result: should be Just 15
result3 :: Maybe Int
result3 = fromYoneda mapped

-- Chain of fmaps: only ONE actual fmap on the underlying Maybe
chained :: Maybe Int
chained = fromYoneda (fmap (*2) (fmap (+3) (toYoneda (Just 4))))
-- Equivalent to: fmap (*2) (fmap (+3) (Just 4)) = fmap (*2) (Just 7) = Just 14
-- But with Yoneda: only one fmap call at the end!

print result3
print chained


Line 20: Functor law
Found:
fmap (* 2) (fmap (+ 3) (toYoneda (Just 4)))
Why not:
fmap ((* 2) . (+ 3)) (toYoneda (Just 4))

Just 15

Just 14

## Coyoneda: свободный функтор

`Coyoneda F a = exists b. (b -> a, F b)` — это свободный функтор над F.

Любой тип F можно сделать функтором через Coyoneda, даже если у F нет экземпляра Functor!

In [4]:
-- Coyoneda as existential type (using ExistentialQuantification)
data Coyoneda f a = forall b. Coyoneda (b -> a) (f b)

-- liftCoyoneda: wrap F a in Coyoneda (use identity function)
liftCoyoneda :: f a -> Coyoneda f a
liftCoyoneda fa = Coyoneda id fa

-- lowerCoyoneda: unwrap requires Functor f
lowerCoyoneda :: Functor f => Coyoneda f a -> f a
lowerCoyoneda (Coyoneda g fb) = fmap g fb

-- Functor instance for Coyoneda (does NOT require Functor f!)
instance Functor (Coyoneda f) where
  fmap h (Coyoneda g fb) = Coyoneda (h . g) fb

-- Test: use Coyoneda to fmap over [] without using [] Functor directly
coList :: Coyoneda [] Int
coList = liftCoyoneda [1, 2, 3]

-- fmap (+10) via Coyoneda (no Functor [] used yet)
mappedCo :: Coyoneda [] Int
mappedCo = fmap (+10) coList

-- Lower: only NOW we use Functor []
result4 :: [Int]
result4 = lowerCoyoneda mappedCo

print result4


Line 6: Eta reduce
Found:
liftCoyoneda fa = Coyoneda id fa
Why not:
liftCoyoneda = Coyoneda id

[11,12,13]

## Практика: ускорение fmap через Yoneda

Если мы делаем много `fmap` подряд, Yoneda позволяет скомпозировать ВСЕ функции в ОДНУ, и применить только раз.

In [5]:
-- Naive repeated fmap on list: each fmap traverses the list
slowFmaps :: [Int] -> [Int]
slowFmaps xs = fmap (*3) (fmap (+1) (fmap (*2) xs))

-- Via Yoneda: compose functions first, then one fmap
fastFmaps :: [Int] -> [Int]  
fastFmaps xs = fromYoneda $ fmap (*3) $ fmap (+1) $ fmap (*2) $ toYoneda xs

-- Both should give same result
testList :: [Int]
testList = [1..5]

slowResult :: [Int]
slowResult = slowFmaps testList

fastResult :: [Int]
fastResult = fastFmaps testList

print slowResult
print fastResult
print (slowResult == fastResult)


Line 3: Functor law
Found:
fmap (* 3) (fmap (+ 1) (fmap (* 2) xs))
Why not:
fmap ((* 3) . (+ 1)) (fmap (* 2) xs)Line 3: Functor law
Found:
fmap (+ 1) (fmap (* 2) xs)
Why not:
fmap ((+ 1) . (* 2)) xsLine 7: Functor law
Found:
fmap (* 3) $ fmap (+ 1) $ fmap (* 2) $ toYoneda xs
Why not:
fmap ((* 3) . (+ 1)) (fmap (* 2) $ toYoneda xs)Line 7: Functor law
Found:
fmap (+ 1) $ fmap (* 2) $ toYoneda xs
Why not:
fmap ((+ 1) . (* 2)) (toYoneda xs)Line 7: Use <$>
Found:
fmap (* 2) $ toYoneda xs
Why not:
((* 2) <$> toYoneda xs)

[9,15,21,27,33]

[9,15,21,27,33]

True

## Представимые функторы

`F ≅ (r ->)` для некоторого `r`. Функтор F представим, если есть изоморфизм между `F a` и `r -> a`.

Пример: `Pair a = (a, a)` представим через `Bool -> a`.

![Представимые функторы](../diagrams/yoneda/yoneda_representable.svg)

In [6]:
-- Pair functor is representable by Bool
-- Pair a ~= Bool -> a
data Pair a = Pair a a deriving (Show)

instance Functor Pair where
  fmap f (Pair x y) = Pair (f x) (f y)

-- tabulate: (Bool -> a) -> Pair a
tabulatePair :: (Bool -> a) -> Pair a
tabulatePair f = Pair (f False) (f True)

-- index: Pair a -> Bool -> a
indexPair :: Pair a -> Bool -> a
indexPair (Pair x _) False = x
indexPair (Pair _ y) True  = y

-- Test round-trip
testPair :: Pair Int
testPair = Pair 10 20

-- index then tabulate: should recover original
roundTrip :: Pair Int
roundTrip = tabulatePair (indexPair testPair)

-- tabulate a function
fromFn :: Pair Int
fromFn = tabulatePair (\b -> if b then 100 else 0)

print testPair
print roundTrip
print fromFn

-- indexPair acts like lookup
print (indexPair testPair False)
print (indexPair testPair True)


Pair 10 20

Pair 10 20

Pair 0 100

10

20

## Связь с расширениями Кана

Лемма Ёнеды — частный случай правого расширения Кана:
`Yoneda f a = Ran Identity f a = forall b. (a -> b) -> f b`

In [7]:
-- Yoneda as a special case of Ran over Identity
-- Ran Identity F a = forall b. (a -> Identity b) -> F b
--                  = forall b. (a -> b) -> F b
--                  = Yoneda F a

-- Concrete demonstration: Ran Identity [] Int
-- = forall b. (Int -> b) -> [b]
-- This is just: given a function Int -> b, produce [b]
-- If we have [Int] = [1,2,3], this is fmap over it!

-- ranIdentityList :: forall b. (Int -> b) -> [b]
-- ranIdentityList f = fmap f [1,2,3]

-- In Yoneda form:
yonedaList :: Yoneda [] Int
yonedaList = toYoneda [1, 2, 3]

-- Running with show gives ["1","2","3"]
result5 :: [String]
result5 = runYoneda yonedaList show

-- Running with (*2) gives [2,4,6]
result6 :: [Int]
result6 = runYoneda yonedaList (*2)

print result5
print result6


["1","2","3"]

[2,4,6]

## Итоги

Лемма Ёнеды говорит, что функтор полностью определяется своими морфизмами из/в него.
Практически: Yoneda даёт нам «бесплатный» fmap и оптимизацию слияния функций.

In [8]:
-- Natural transformation from (Int ->) to []
-- Nat((Int ->), []) ~= [Int]   (by Yoneda)
-- A nat. trans. alpha :: forall b. (Int -> b) -> [b]
-- is determined by alpha id :: [Int]

-- Example nat. trans.: replicate
replicateNat :: forall b. (Int -> b) -> [b]
replicateNat f = fmap f [1, 2, 3]

-- By Yoneda, this is determined by replicateNat id = [1,2,3]
yonedaElement :: [Int]
yonedaElement = replicateNat id

-- Recovering the nat. trans. from the element [1,2,3]
recovered :: forall b. (Int -> b) -> [b]
recovered = \f -> fmap f yonedaElement

-- Both should give same results
test1 :: [String]
test1 = replicateNat show

test2 :: [String]
test2 = recovered show

print yonedaElement
print test1
print test2
print (test1 == test2)


Line 16: Redundant lambda
Found:
recovered = \ f -> fmap f yonedaElement
Why not:
recovered f = fmap f yonedaElementLine 16: Avoid lambda using `infix`
Found:
\ f -> fmap f yonedaElement
Why not:
(`fmap` yonedaElement)

[1,2,3]

["1","2","3"]

["1","2","3"]

True

## Заключение

- `Yoneda F a ~= F a`: изоморфизм позволяет отложить `fmap`
- `Functor (Yoneda f)` не требует `Functor f`
- Coyoneda делает любой тип функтором (свободный функтор)
- Представимые функторы: `F ~= (r ->)` — функторы с индексированным доступом
- Связь с Kan: `Yoneda F = Ran Identity F`
- Лемма Ёнеды — основа профунктор-оптик и расширений Кана

## Диаграмма: лемма Ёнеды

Изоморфизм Nat(Hom(a,−), F) ≅ F a.

![Лемма Ёнеды](../diagrams/yoneda/yo_yoneda.svg)

---

**← Предыдущий:** [Оптики](Optics.ipynb)  |  **Следующий →** [Расширения Кана](KanExtensions.ipynb)
